# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Gather record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print('No record sets found in the dataset metadata. Please check the schema definitions.')
else:
    print('Record Set @ids:')
    for rs_id in record_set_ids:
        print(f'  - {rs_id}')

# List fields for each record set by @id
for rs_id in record_set_ids:
    print(f"\nRecord Set '{rs_id}' fields:")
    record_set_obj = None
    # Locate the record set dict in metadata
    for rs in dataset.metadata.to_json().get('recordSet', []):
        if rs.get('@id', None) == rs_id:
            record_set_obj = rs
            break
    if not record_set_obj:
        print('  Could not find record set object in metadata JSON.')
        continue
    # List field @ids
    fields = record_set_obj.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, str):
            print(f'    - {field}')
        elif isinstance(field, dict):
            print(f"    - {field.get('@id', '(unknown)')} ({field.get('name', '')})")
        else:
            print(f"    - Unknown field structure: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Discover record sets in the metadata
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  DataFrame shape: {dataframes[record_set_id].shape}")
    print(f"  Sample columns: {dataframes[record_set_id].columns.tolist()}")
    print(dataframes[record_set_id].head(3))

# If there is only one record_set, use it for further analysis.
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"\nUsing record set '{main_rs}' for further analysis.")
    print(f"Columns: {dataframes[main_rs].columns.tolist()}")
    display(dataframes[main_rs].head())
else:
    print('No record set available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Identify a likely numeric field (try a handful common for protein quantification)
df = dataframes[main_rs]
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
if not numeric_candidates:
    # Try to infer numerics if all string (mass spec data can have numeric as strings)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
                numeric_candidates.append(col)
        except Exception:
            continue

if not numeric_candidates:
    print('No numeric field found for EDA. Skipping analysis.')
else:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() # Use mean as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by 'description' or another categorical, if available
    group_candidates = [col for col in filtered_df.columns if filtered_df[col].dtype == 'O'] # object cols
    group_field = group_candidates[0] if group_candidates else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    # Plot the distribution of the main numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there's a group field, plot boxplot as well
    if group_field is not None:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 mass spectrometry dataset, examined its metadata, explored available record sets and their fields by `@id`, extracted a main record set into a DataFrame, and performed basic exploratory analysis and visualization. This approach can be extended for deeper biological or proteomics insights, such as more advanced filtering or integration with protein knowledge bases.

For best reproducibility, always reference fields and entities by their Croissant `@id` when accessing or processing data. For further analyses, consult the full Croissant metadata and schema.